# 25 · Sensor: подтипы отказа

Задача «Отказ датчика» сегодня использует ОБОБЩЁННОЕ событие «неисправность»
(любой статус из `Неисправен / Обесточен / Отключено устройство / Затоплен`).
Разберём подтипы:
1. (А) распределение подтипов по типам датчиков (из `value_by_type`);
2. (Б) агрегируем журнал по (канал, бакет) с разделением по подтипам
   (это будущие колонки панели для мультиподтиповой модели);
3. (В) обучаем TTE fire-style для основных подтипов отдельно и сравниваем,
   стоит ли дробить таргет (если подтипы статистически различимы и
   предсказуемы — да; если нет — оставляем обобщённый).


In [ ]:
%matplotlib inline
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd

import data_utils as du
import features as fe
import tte_pipeline as tte
from tte_experiments import survival_metrics, pr_at_horizon, fit_discrete_hazard


DATASET = du.find_dataset_dir()
S_TYPE = ["Датчик дыма", "Датчик движения", "Датчик температуры", "Газовый датчик",
          "КД Дверь", "КД АВ", "КД Люк", "Тепловой датчик", "Стекло",
          "Датчик затопления"]
SUBTYPES = ["Неисправен", "Обесточен", "Отключено устройство", "Затоплен",
            "Не замкнут"]
ref = du.load_ref_channels()[["ид_канала_данных", "тип_датчика"]]
ch_set = set(ref.loc[ref["тип_датчика"].isin(S_TYPE), "ид_канала_данных"])
print("каналов sensor-типов:", len(ch_set))


In [ ]:
# (А) Распределение подтипов по типам датчиков
v = pd.read_csv(DATASET / "value_by_type.csv", encoding="utf-8-sig",
                dtype=str)
v = v[v["значение"].isin(SUBTYPES)].copy()
v["n"] = pd.to_numeric(v["n"], errors="coerce")
tab = v.pivot_table(index="тип_датчика", columns="значение", values="n",
                    aggfunc="sum", fill_value=0)
tab["ИТОГО"] = tab.sum(axis=1)
print(tab.to_string())
print()
print("Доли подтипов среди отказов (суммарно):")
tot = v.groupby("значение")["n"].sum()
print((100 * tot / tot.sum()).round(2).to_string())


In [ ]:
# (Б) Агрегаты подтипов по (канал, 6ч бакет) — проход по журналам (~10-15 мин)
hour_ns = fe.BUCKET_HOURS * 3600 * 10 ** 9
parts = []
for fp in du.journal_files():
    for ch in du.read_chunks(fp, cols=["ид_канала_данных", "дата", "время",
                                       "значение_датчика"], chunksize=500_000):
        m = ch["ид_канала_данных"].isin(ch_set)
        if not m.any():
            continue
        ch = ch[m]
        v = ch["значение_датчика"].fillna("<нет>").astype(str)
        subm = v.isin(SUBTYPES)
        if not subm.any():
            continue
        ch = ch[subm]
        ts = pd.to_datetime(ch["дата"] + " " + ch["время"].fillna("00:00:00"),
                            format="%Y-%m-%d %H:%M:%S", errors="coerce")
        ch = ch[ts.notna()]
        if not ch.empty:
            buck = (ts[ts.notna()].astype("datetime64[ns]").astype("int64") // hour_ns).to_numpy()
            dfc = pd.DataFrame({"ид": ch["ид_канала_данных"].astype(str).to_numpy(),
                                "бакет": buck, "st": v[subm].to_numpy()})
            dfc["пол"] = 1
            piv = dfc.pivot_table(index=["ид", "бакет"], columns="st",
                                  values="пол", aggfunc="sum", fill_value=0)
            parts.append(piv.reset_index())
    print(f"  {fp.name}: строк={sum(len(p) for p in parts):,}", flush=True)

sub = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
sub = sub.groupby(["ид", "бакет"], as_index=False)[SUBTYPES].sum()
sub.to_parquet(DATASET / "sensor_subtype_bucket.parquet")
print("sensor_subtype_bucket:", sub.shape)
print((sub[SUBTYPES] > 0).sum().to_string())


In [ ]:
# (В) Присоединяем подтипы к sensor-панели и считаем метрики по главным подтипам
panel = pd.read_csv(DATASET / "subdaily_panel_sensor6h.csv",
                    dtype={"ид_канала_данных": str, "ид_объект": str})
sub = pd.read_parquet(DATASET / "sensor_subtype_bucket.parquet")
sub["ид"] = sub["ид"].astype(str)
panel = panel.merge(sub.rename(columns={"ид": "ид_канала_данных"}),
                    on=["ид_канала_данных", "бакет"], how="left")
panel[SUBTYPES] = panel[SUBTYPES].fillna(0)
panel["неисправностей_неиспр"] = panel["Неисправен"]
panel["неисправностей_обес"] = panel["Обесточен"]
print("panel:", panel.shape)
for st in ["неисправностей_неиспр", "неисправностей_обес"]:
    print(st, "база на панели {:.4f}%".format((panel[st] > 0).mean() * 100))


In [ ]:
def run_sub(col, tag):
    p = tte.refit_z_train_only(panel.copy())
    p = tte.add_series_context(p, event_col=col)
    s = tte.add_tte_labels(p)
    s["ид_объект_code"] = s["ид_объект"].astype("category").cat.codes
    s["тип_датчика_code"] = s["тип_датчика"].astype("category").cat.codes
    tr, va, ho = tte.split_subjects(s)
    ev = tr["event_flag"].fillna(0).to_numpy(); od = tr["obs_days"].to_numpy()
    print(f"[{tag}] train база <=1д: {((ev==1)&(od<=1.01)).mean()*100:.3f}% | "
          f"серий: {int(((ev==1)&(od<=1.01)).sum())}", flush=True)
    Xc = tte.subject_features(tr)
    trs = tte.sample_subjects(tr, 12000, 8000, 8000, seed=42)
    vas = tte.sample_subjects(va, 2000, 2000, 2000, seed=43)
    hos = tte.sample_holdout(ho, 20000, seed=777)
    X_tr, y_tr = tte.expand_person_time(trs, Xc, tte.HORIZON_BUCKETS)
    X_va, y_va = tte.expand_person_time(vas, Xc, tte.HORIZON_BUCKETS)
    m = fit_discrete_hazard(X_tr, y_tr, X_va, y_va, iters=700, lr=0.03,
                            depth=10, seed=42, task_type="GPU")
    surv_tr = tte.make_surv_struct(trs["event_flag"], trs["obs_days"])
    surv_ho = tte.make_surv_struct(hos["event_flag"], hos["obs_days"])
    Xh, _ = tte.expand_person_time(hos, Xc, tte.HORIZON_BUCKETS, fixed_horizon=True)
    S = np.cumprod(1 - m.predict_proba(Xh)[:, 1].reshape(-1, tte.HORIZON_BUCKETS), axis=1)
    met = survival_metrics(surv_tr, surv_ho, 1 - S[:, -1], S, tte.EVAL_TIMES)
    y24 = ((hos["event_flag"] == 1) & (hos["obs_days"] <= 1.01)).to_numpy(int)
    p24p = 1 - S[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
    pr = pr_at_horizon(y24, p24p)
    print(f"[{tag}] Uno-C={met['uno_c']:.3f} | IBS={met['ibs']:.3f} | "
          f"P(24ч) PR-AUC={pr['pr_auc']:.3f} (база {pr['base']*100:.2f}%)", flush=True)

run_sub("неисправностей_неиспр", "подтип Неисправен")
run_sub("неисправностей_обес", "подтип Обесточен")


In [ ]:
print("Для сравнения — обобщённый отказа (любой статус, новые дефолты):")
print("  Uno-C=0.710 | PR-AUC(24ч)=0.016 | база 0.31%")
